In [12]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pickle

In [ ]:
r1 = pd.read_csv('Data/100_sgn_metric_list_hist_DNetFinder.csv')
r2 = pd.read_csv('Data/200_sgn_metric_list_hist_DNetFinder.csv')

s1 = pd.read_csv('Data/100_sgn_metric_list_hist_DiffNetFDR.csv')
s2 = pd.read_csv('Data/200_sgn_metric_list_hist_DiffNetFDR.csv')


with open('Data/our_method_metric_list', 'rb') as f:
    our_method_metric_list = pickle.load(f)

runs_precision_list = our_method_metric_list[0]
runs_recall_list = our_method_metric_list[1]
runs_fdr_list = our_method_metric_list[2]
runs_qs_list = our_method_metric_list[3]
runs_time_mean_list = our_method_metric_list[4]
runs_time_std_list = our_method_metric_list[5]

In [ ]:
# Construct markdown table for Mean ± SD (sample standard deviation)

table = f"""
| Dimension | DiffNetFDR_Xia2015 | DNetFinder_Liu2017 | DNetEx |
|---|---|---|---|
| 100 | {s1.groupby('alpha').diffnetfinder_sec.agg(['mean', 'std']).reset_index()['mean'][0]:.4f} ± {s1.groupby('alpha').diffnetfinder_sec.agg(['mean', 'std']).reset_index()['std'][0]:.4f} | {r1.groupby('alpha').dnetfinder_sec.agg(['mean', 'std']).reset_index()['mean'][0]:.4f} ± {r1.groupby('alpha').dnetfinder_sec.agg(['mean', 'std']).reset_index()['std'][0]:.4f} | {runs_time_mean_list[0][0]:.4f} ± {runs_time_std_list[0][0]:.4f} |
| 200 | {s2.groupby('alpha').diffnetfinder_sec.agg(['mean', 'std']).reset_index()['mean'][0]:.4f} ± {s2.groupby('alpha').diffnetfinder_sec.agg(['mean', 'std']).reset_index()['std'][0]:.4f} | {r2.groupby('alpha').dnetfinder_sec.agg(['mean', 'std']).reset_index()['mean'][0]:.4f} ± {r2.groupby('alpha').dnetfinder_sec.agg(['mean', 'std']).reset_index()['std'][0]:.4f} | {runs_time_mean_list[1][0]:.4f} ± {runs_time_std_list[1][0]:.4f} |
"""

print(table)


| Dimension | DiffNetFDR_Xia2015 | DNetFinder_Liu2017 | DNetEx |
|---|---|---|---|
| 100 | 0.1841 ± 0.0145 | 0.2332 ± 0.0083 | 0.0415 ± 0.0063 |
| 200 | 0.5466 ± 0.0072 | 0.9401 ± 0.0853 | 0.1421 ± 0.0249 |



In [43]:
pfig = go.Figure()
temp_df = r1.groupby('alpha').FDR.mean().reset_index()
temp_df2 = s1.groupby('alpha').FDR.mean().reset_index()
grouped = r1.groupby('alpha').FDR.agg(['mean', 'std']).reset_index()

run_number = 0

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=temp_df.alpha.values,
        y=temp_df.FDR.values,
        mode='lines+markers',
        name='DNetFinder, p=100'
    )
)

fig.add_trace(
    go.Scatter(
        x=temp_df2.alpha.values,
        y=temp_df2.FDR.values,
        mode='lines+markers',
        name='DiffNetFDR, p=100'
    )
)

fig.add_trace(
    go.Scatter(
        x=runs_qs_list[run_number],
        y=runs_fdr_list[run_number],
        mode='lines+markers',
        name=f'DNetEx, p=100'
    )
)

temp_df = r2.groupby('alpha').FDR.mean().reset_index()
temp_df2 = s2.groupby('alpha').FDR.mean().reset_index()
grouped = r2.groupby('alpha').FDR.agg(['mean', 'std']).reset_index()

run_number = 1

fig.add_trace(
    go.Scatter(
        x=temp_df.alpha.values,
        y=temp_df.FDR.values,
        mode='lines+markers',
        name='DNetFinder, p=200'
    )
)

fig.add_trace(
    go.Scatter(
        x=temp_df2.alpha.values,
        y=temp_df2.FDR.values,
        mode='lines+markers',
        name='DiffNetFDR, p=200'
    )
)

fig.add_trace(
    go.Scatter(
        x=runs_qs_list[run_number],
        y=runs_fdr_list[run_number],
        mode='lines+markers',
        name=f'DNetEx, p=200'
    )
)

fig.add_trace(
    go.Scatter(
        x=[min(runs_qs_list[run_number]), max(runs_qs_list[run_number])],
        y=[min(runs_qs_list[run_number]), max(runs_qs_list[run_number])],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name="f(x) = x"
    )
)

fig.update_traces(line=dict(width=3))

fig.update_layout(
    autosize=False,
    xaxis1=dict(tickfont=dict(size=16),showticklabels=True),
    yaxis1=dict(tickfont=dict(size=16),showticklabels=True),
    width=800,
    height=500,
    xaxis_title='Control FDR (q)',
    yaxis_title='Real FDR',
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=50, r=50, t=50, b=50),
    showlegend=True,
    legend=dict(
        x=0.005,
        y=0.99,  
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.6)',
        bordercolor='gray',
        borderwidth=1
    ),
    xaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    yaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    font=dict(size=16)
)

fig.write_image("Results/fdr-control.pdf")
fig.show()

# Precision-Recall

In [44]:
temp_df = r1.groupby('alpha').aggregate({'NRecall':'mean','FDR':'mean'}).reset_index()
temp_df2 = s1.groupby('alpha').aggregate({'NRecall':'mean','FDR':'mean'}).reset_index()
run_number = 0

# Create figure
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=temp_df.NRecall.values,
        y=1-temp_df.FDR.values,
        mode='lines+markers',
        name='DNetFinder, p=100'
    )
)

fig.add_trace(
    go.Scatter(
        x=temp_df2.NRecall.values,
        y=1-temp_df2.FDR.values,
        mode='lines+markers',
        name='DiffNetFDR, p=100'
    )
)

fig.add_trace(
    go.Scatter(
        x=runs_recall_list[run_number],
        y=1-runs_fdr_list[run_number],
        mode='lines+markers',
        name=f'DNetEx, p=100'
    )
)

temp_df = r2.groupby('alpha').aggregate({'NRecall':'mean','FDR':'mean'}).reset_index()
temp_df2 = s2.groupby('alpha').aggregate({'NRecall':'mean','FDR':'mean'}).reset_index()
run_number = 1

fig.add_trace(
    go.Scatter(
        x=temp_df.NRecall.values,
        y=1-temp_df.FDR.values,
        mode='lines+markers',
        name='DNetFinder, p=200'
    )
)

fig.add_trace(
    go.Scatter(
        x=temp_df2.NRecall.values,
        y=1-temp_df2.FDR.values,
        mode='lines+markers',
        name='DiffNetFDR, p=200'
    )
)

fig.add_trace(
    go.Scatter(
        x=runs_recall_list[run_number],
        y=1-runs_fdr_list[run_number],
        mode='lines+markers',
        name=f'DNetEx, p=200'
    )
)


# Customize the layout
fig.update_traces(line=dict(width=3))

fig.update_layout(
    xaxis1=dict(tickfont=dict(size=16),showticklabels=True),
    yaxis1=dict(tickfont=dict(size=16),showticklabels=True),
    autosize=False,
    width=800,
    height=500,
    xaxis_title=r"Recall",
    yaxis_title='Precision',
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=50, r=50, t=50, b=50),
    showlegend=True,
    legend=dict(
        x=.995,
        y=0.99,  
        xanchor='right',
        yanchor='top',
        bgcolor='white',
        bordercolor='gray',
        borderwidth=1
    ),
    xaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    yaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    font=dict(size=16)

)

fig.show()

fig.write_image("Results/pr.pdf")
